In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files (they are in the working directory)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Convert timestamp to timezone-aware datetimes (UTC->Asia/Shanghai) if timestamp column exists
for df in (df_metric, df_trace, df_log, df_error):
    if 'timestamp' in df.columns:
        # create a tz-aware datetime column named 'ts'
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# METRIC summary
metric_total_rows = df_metric.shape[0]
metric_cmdbs = sorted(df_metric['cmdb_id'].dropna().unique().tolist())
metric_top_pairs = (
    df_metric.groupby(['cmdb_id', 'kpi_name']).size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

metric_sample = df_metric.head(5)  # first 5 rows to confirm schema

# TRACE summary
trace_total_rows = df_trace.shape[0]
trace_cmdbs = sorted(df_trace['cmdb_id'].dropna().unique().tolist())
trace_top_pairs = (
    df_trace.groupby(['cmdb_id', 'trace_name']).size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# LOG summary
log_total_rows = df_log.shape[0]
log_cmdbs = sorted(df_log['cmdb_id'].dropna().unique().tolist())
log_top_pairs = (
    df_log.groupby(['cmdb_id', 'log_name']).size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# ERROR_LOGS summary
error_total_rows = df_error.shape[0]
error_cmdbs = sorted(df_error['cmdb_id'].dropna().unique().tolist())

# Use message snippet (first 200 chars) to compute distinct snippets per cmdb_id
if 'message' in df_error.columns:
    df_error['msg_snip'] = df_error['message'].fillna('').astype(str).str.slice(0, 200)
    error_counts = (
        df_error.groupby('cmdb_id')
        .agg(total_count=('message', 'size'),
             distinct_msg_snippets=('msg_snip', 'nunique'))
        .reset_index()
        .sort_values(['total_count', 'distinct_msg_snippets'], ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    # fallback if no message column: just counts per cmdb_id
    error_counts = (
        df_error.groupby('cmdb_id').size().reset_index(name='total_count').sort_values('total_count', ascending=False).head(20)
    )

# Compact textual summaries and samples (variables displayed)
metric_total_rows, metric_cmdbs, metric_top_pairs[['cmdb_id','kpi_name','count']], metric_sample, \
trace_total_rows, trace_cmdbs, trace_top_pairs[['cmdb_id','trace_name','count']], \
log_total_rows, log_cmdbs, log_top_pairs[['cmdb_id','log_name','count']], \
error_total_rows, error_cmdbs, error_counts

```
Out[1]:
```
```python
summary = (
"Telemetry summary (timestamps in UTC+8):\n\n"
"1) metric.csv\n"
"- Total rows: 27907\n"
"- Unique cmdb_id values (all): ['IG01', 'IG02', 'MG01', 'MG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02',\n"
"  'ServiceTest1', 'ServiceTest10', 'ServiceTest11', 'ServiceTest2', 'ServiceTest3', 'ServiceTest4',\n"
"  'ServiceTest5', 'ServiceTest6', 'ServiceTest7', 'ServiceTest8', 'ServiceTest9', 'Tomcat01',\n"
"  'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2']\n"
"- Top 20 (cmdb_id, kpi_name, count):\n"
"  1. (IG01, JVM-Threads_7778_JVM_ThreadCount_Threads) — 31\n"
"  2. (Mysql01, OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount) — 31\n"
"  3. (IG01, OSLinux-CPU_CPU_CPUSysTime) — 31\n"
"  4. (Mysql01, OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies) — 31\n"
"  5. (Mysql01, OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...) — 31\n"
"  6. (apache02, OSLinux-OSLinux_ZABBIX_Host_Uptime) — 31\n"
"  7. (apache02, OSLinux-OSLinux_PROCESS_apache_10001_PROCPPCount) — 31\n"
"  8. (apache02, OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...) — 31\n"
"  9. (apache02, OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount) — 31\n"
" 10. (IG01, OSLinux-CPU_CPU_CPUidleutil) — 31\n"
" 11. (IG01, OSLinux-CPU_CPU_CPULoad) — 31\n"
" 12. (IG01, OSLinux-CPU_CPU_CPUUserTime) — 31\n"
" 13. (IG01, OSLinux-CPU_CPU_CPUWio) — 31\n"
" 14. (IG01, JVM-Memory_7778_JVM_Memory_HeapMemoryMax) — 31\n"
" 15. (IG01, JVM-Memory_7778_JVM_Memory_HeapMemoryUsed) — 31\n"
" 16. (IG01, JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed) — 31\n"
" 17. (IG01, OSLinux-OSLinux_ZABBIX_Host_Uptime) — 31\n"
" 18. (Tomcat04, Tomcat-Sessions_7441--_SESSIONRejectedSessions) — 31\n"
" 19. (Tomcat04, Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...) — 31\n"
" 20. (Tomcat04, Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...) — 31\n"
"- Sample first 5 rows (confirming schema):\n"
"  timestamp, cmdb_id, kpi_name, value, ts\n"
"  1615276800, IG01, JVM-Memory_7778_JVM_Memory_HeapMemoryMax, 2.040136e+09, 2021-03-09 16:00:00+08:00\n"
"  1615276800, IG01, JVM-Memory_7778_JVM_Memory_HeapMemoryUsed, 4.101854e+08, 2021-03-09 16:00:00+08:00\n"
"  1615276800, IG01, JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed, 5.791533e+07, 2021-03-09 16:00:00+08:00\n"
"  1615276800, IG01, JVM-Threads_7778_JVM_ThreadCount_Threads, 67.0, 2021-03-09 16:00:00+08:00\n"
"  1615276800, IG01, OSLinux-CPU_CPU-0_SingleCpuUtil, 1.8475, 2021-03-09 16:00:00+08:00\n\n"
"2) trace.csv\n"
"- Total rows: 6936\n"
"- Unique cmdb_id values: ['IG01', 'IG02', 'MG01', 'MG02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2']\n"
"- Top 20 (cmdb_id, trace_name, count):\n"
"  1-3. (dockerB2, trace.to_MG02.row_count/duration_p95/duration_mean) — 30 each\n"
"  4. (dockerB2, trace.to_MG01.row_count) — 30\n"
"  5. (IG01, trace.self.duration_mean) — 30\n"
"  6-11. (dockerA2, various trace.to/from/MG01/MG02 metrics and self.*) — 30 each\n"
" 12-20. (dockerB1/dockerA2/dockerA1 entries for various trace metrics) — 30 each\n\n"
"3) log.csv\n"
"- Total rows: 511\n"
"- Unique cmdb_id values: ['Mysql01', 'Mysql02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02']\n"
"- Top 20 (cmdb_id, log_name, count):\n"
"  1. (apache02, log.error_count) — 31\n"
"  2. (Tomcat03, log.error_count) — 31\n"
"  3. (Tomcat01, log.row_count) — 31\n"
"  4. (Tomcat01, log.error_count) — 31\n"
"  5. (apache02, log.row_count) — 31\n"
"  6. (apache01, log.row_count) — 31\n"
"  7. (Tomcat04, log.error_count) — 31\n"
"  8. (Tomcat03, log.row_count) — 31\n"
"  9. (apache01, log.error_count) — 31\n"
" 10. (Tomcat04, log.row_count) — 31\n"
" 11-19. (Mysql01/Mysql02 entries and various Tomcat gc_pause/log.error_count rows) — counts between 30 and 5\n\n"
"4) error_logs.csv\n"
"- Total rows: 1\n"
"- Unique cmdb_id values: ['apache02']\n"
"- For apache02: total_count = 1, distinct message snippets = 1\n\n"
"Notes: - All outputs are compact summaries. For fuller details, refer to the displayed DataFrames from the earlier execution.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(27907, ['IG01', 'IG02', 'MG01', 'MG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02', 'ServiceTest1', 'ServiceTest10', 'ServiceTest11', 'ServiceTest2', 'ServiceTest3', 'ServiceTest4', 'ServiceTest5', 'ServiceTest6', 'ServiceTest7', 'ServiceTest8', 'ServiceTest9', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'],      cmdb_id                                           kpi_name  count
0       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31
1    Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31
2       IG01                         OSLinux-CPU_CPU_CPUSysTime     31
3    Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31
4    Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31
5   apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31
6   apache02   OSLinux-OSLinux_PROCESS_apache_10001_PROCPPCount     31
7   apache02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31
8   apache02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31
9       IG01                        OSLinux-CPU_CPU_CPUidleutil     31
10      IG01                            OSLinux-CPU_CPU_CPULoad     31
11      IG01                        OSLinux-CPU_CPU_CPUUserTime     31
12      IG01                             OSLinux-CPU_CPU_CPUWio     31
13      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31
14      IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31
15      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31
16      IG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31
17  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31
18  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31
19  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31,     timestamp cmdb_id                                     kpi_name         value                        ts
0  1615276800    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09 2021-03-09 16:00:00+08:00
1  1615276800    IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  4.101854e+08 2021-03-09 16:00:00+08:00
2  1615276800    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.791533e+07 2021-03-09 16:00:00+08:00
3  1615276800    IG01     JVM-Threads_7778_JVM_ThreadCount_Threads  6.700000e+01 2021-03-09 16:00:00+08:00
4  1615276800    IG01              OSLinux-CPU_CPU-0_SingleCpuUtil  1.847500e+00 2021-03-09 16:00:00+08:00, 6936, ['IG01', 'IG02', 'MG01', 'MG02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'],      cmdb_id                     trace_name  count
0   dockerB2        trace.to_MG02.row_count     30
1   dockerB2     trace.to_MG02.duration_p95     30
2   dockerB2    trace.to_MG02.duration_mean     30
3   dockerB2        trace.to_MG01.row_count     30
4       IG01       trace.self.duration_mean     30
5   dockerA2           trace.self.row_count     30
6   dockerA2    trace.to_MG01.duration_mean     30
7   dockerA2     trace.to_MG01.duration_p95     30
8   dockerA2        trace.to_MG01.row_count     30
9   dockerA2    trace.to_MG02.duration_mean     30
10  dockerA2     trace.to_MG02.duration_p95     30
11  dockerA2        trace.to_MG02.row_count     30
12  dockerB1  trace.from_MG01.duration_mean     30
13  dockerA2  trace.from_MG02.duration_mean     30
14  dockerA2   trace.from_MG02.duration_p95     30
15  dockerA2      trace.from_MG02.row_count     30
16  dockerA2       trace.self.duration_mean     30
17  dockerA2        trace.self.duration_p95     30
18  dockerA1     trace.to_MG01.duration_p95     30
19  dockerA1        trace.to_MG01.row_count     30, 511, ['Mysql01', 'Mysql02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02'],      cmdb_id          log_name  count
0   apache02   log.error_count     31
1   Tomcat03   log.error_count     31
2   Tomcat01     log.row_count     31
3   Tomcat01   log.error_count     31
4   apache02     log.row_count     31
5   apache01     log.row_count     31
6   Tomcat04   log.error_count     31
7   Tomcat03     log.row_count     31
8   apache01   log.error_count     31
9   Tomcat04     log.row_count     31
10   Mysql01     log.row_count     30
11   Mysql01   log.error_count     30
12  Tomcat01  log.gc_pause_max     22
13  Tomcat02     log.row_count     22
14  Tomcat03  log.gc_pause_max     22
15  Tomcat02   log.error_count     22
16  Tomcat02  log.gc_pause_max     22
17  Tomcat04  log.gc_pause_max     21
18   Mysql02     log.row_count      5
19   Mysql02   log.error_count      5, 1, ['apache02'],     cmdb_id  total_count  distinct_msg_snippets
0  apache02            1                      1)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse df_metric loaded earlier in the session
# Restrict to specified cmdb_ids
allowed_cmdbs = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
                 'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

df_m = df_metric[df_metric['cmdb_id'].isin(allowed_cmdbs)].copy()

# 2) Compute global stats (using entire series for each (cmdb_id,kpi_name))
global_stats = (
    df_m.groupby(['cmdb_id','kpi_name'])['value']
    .agg(global_count='size',
         global_p95=lambda x: x.quantile(0.95),
         global_mean='mean',
         global_max='max')
    .reset_index()
)

# 3) Filter metric rows to incident window (inclusive)
start_ts = 1615276800
end_ts = 1615278600
df_window = df_m[(df_m['timestamp'] >= start_ts) & (df_m['timestamp'] <= end_ts)].copy()

# 4) For each series, detect anomalies where value > global_p95
# Merge global stats into window rows so we can compare per-row
df_win_merged = df_window.merge(global_stats, on=['cmdb_id','kpi_name'], how='left')

# boolean flag for anomaly rows
df_win_merged['is_anom'] = df_win_merged['value'] > df_win_merged['global_p95']

# window_max per series (max within window, even if no anomaly)
window_max_df = df_window.groupby(['cmdb_id','kpi_name'])['value'].agg(window_max='max').reset_index()

# anomaly event summary: earliest anomaly timestamp and count of anomaly rows in window
anom_events = (
    df_win_merged[df_win_merged['is_anom']]
    .groupby(['cmdb_id','kpi_name'])['timestamp']
    .agg(earliest_anomaly_timestamp='min', anomaly_count_in_window='size')
    .reset_index()
)

# Merge to form anomaly summary (only series with at least one anomaly)
anomaly_summary = (
    anom_events
    .merge(global_stats, on=['cmdb_id','kpi_name'], how='left')
    .merge(window_max_df, on=['cmdb_id','kpi_name'], how='left')
)

# compute ratio = window_max / global_p95 safely
anomaly_summary['ratio'] = np.where(
    anomaly_summary['global_p95'] == 0,
    np.where(anomaly_summary['window_max'] > 0, np.inf, 1.0),
    anomaly_summary['window_max'] / anomaly_summary['global_p95']
)

# convert earliest_anomaly_timestamp to Asia/Shanghai tz-aware datetime strings
if not anomaly_summary.empty:
    anomaly_summary['earliest_anomaly_ts_local'] = pd.to_datetime(
        anomaly_summary['earliest_anomaly_timestamp'], unit='s', utc=True
    ).dt.tz_convert(tz)

# Sort by ratio descending and prepare final columns
anomaly_summary = anomaly_summary.sort_values('ratio', ascending=False).reset_index(drop=True)
cols = ['cmdb_id','kpi_name','global_count','global_p95','global_max',
        'window_max','earliest_anomaly_ts_local','anomaly_count_in_window','ratio']
result_df = anomaly_summary[cols].head(20)

# 5) If no series exceed p95, provide top 10 series by window_max / global_p95 (even if <=1)
if result_df.shape[0] == 0:
    # Prepare candidates from all series that have window data
    candidates = (
        window_max_df
        .merge(global_stats, on=['cmdb_id','kpi_name'], how='left')
    )
    candidates['ratio'] = np.where(
        candidates['global_p95'] == 0,
        np.where(candidates['window_max'] > 0, np.inf, 1.0),
        candidates['window_max'] / candidates['global_p95']
    )
    # no earliest anomaly timestamp since none exceed p95; keep it NaT
    candidates = candidates.sort_values('ratio', ascending=False).reset_index(drop=True)
    # pick top 10 to help loosen threshold
    result_df = candidates[['cmdb_id','kpi_name','global_count','global_p95','global_max','window_max','ratio']].head(10)

# Keep full intermediate DataFrames in memory (global_stats, df_window, df_win_merged, window_max_df, anom_events, anomaly_summary)
# Display compact textual table (max 20 rows)
result_df

```
Out[2]:
```
summary = (
"Incident-window summary (window 1615276800–1615278600 => 2021-03-09 16:00–16:30 UTC+8):\n\n"
"Overview:\n"
"- 20 (cmdb_id, kpi_name) series exceeded their global 95th-percentile during the window (top 20 returned).\n"
"- Several series have global_p95 == 0 and non-zero values in the window, producing infinite ratio → these are clear outliers.\n\n"
"Top findings (highest priority to investigate):\n"
"1) apache02 – OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT: window value > historic p95 (global_p95=0) at 2021-03-09 16:24, ratio=inf.\n"
"2) Mysql02 – multiple MySQL metrics (Opened table definitions, Table open cache misses, Opened Tables): each had global_p95=0 and window_max >0 at ~16:26, ratio=inf.\n"
"3) Mysql01 – Mysql-MySQL_3306_Innodb pages created: global_p95=0, anomaly at 16:00, ratio=inf.\n"
"4) Tomcat04 – OSLinux NETWORK TCP-CLOSE-WAIT: window_max=1 vs p95=0.05, earliest 16:25, ratio=20.0.\n"
"5) MG01 / MG02 – TCP-FIN-WAIT or TCP-CLOSE-WAIT spikes (ratios 10.0 and 10.0), earliest ~16:25 / 16:06.\n"
"\n"
"Other notable spikes (disk/CPU related):\n"
"- apache02 and IG nodes show local disk DSKRead/DSKPer.. and CPU WIO/CPUUtil spikes with ratios ~3.2–3.33 (examples: IG01 sda DSKRead at 16:08; apache02 CPUWio and disk metrics around 16:25).\n\n"
"Interpretation & next steps:\n"
"- Immediate focus: investigate TCP CLOSE-WAIT/FIN-WAIT on apache02, Tomcat04, MG01/MG02 (socket lifecycle issues) and MySQL anomalies on Mysql02/Mysql01 (table/cache metrics). These are the strongest anomalies (many are historic-p95==0 events).\n"
"- Also review disk I/O and CPU WIO spikes on IG and apache nodes (ratios ~3x), which may contribute to service degradation.\n"
"- Correlate with application logs and connection counts around the listed earliest timestamps (16:00–16:30) to find root cause.\n\n"
"Summary prepared from metric.csv global stats and incident-window comparisons; full intermediate tables are retained in the kernel for follow-up investigation."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  global_count  global_p95  global_max  window_max earliest_anomaly_ts_local  anomaly_count_in_window      ratio
0   apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            31    0.000000    1.000000    1.000000 2021-03-09 16:24:00+08:00                        1        inf
1    Mysql02          Mysql-MySQL_3306_Opened table definitions            21    0.000000    0.175400    0.175400 2021-03-09 16:26:00+08:00                        1        inf
2    Mysql02           Mysql-MySQL_3306_Table open cache misses            21    0.000000    1.701800    1.701800 2021-03-09 16:26:00+08:00                        1        inf
3    Mysql02                     Mysql-MySQL_3306_Opened Tables            21    0.000000    1.701800    1.701800 2021-03-09 16:26:00+08:00                        1        inf
4    Mysql01              Mysql-MySQL_3306_Innodb pages created            21    0.000000    0.016900    0.016900 2021-03-09 16:00:00+08:00                        1        inf
5   Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            20    0.050000    1.000000    1.000000 2021-03-09 16:25:00+08:00                        1  20.000000
6       MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            19    1.000000   10.000000   10.000000 2021-03-09 16:25:00+08:00                        1  10.000000
7       MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            19    0.100000    1.000000    1.000000 2021-03-09 16:06:00+08:00                        1  10.000000
8   apache01                    OSLinux-CPU_CPU-1_SingleCpuidle            16   20.773175   83.092700   83.092700 2021-03-09 16:00:00+08:00                        1   4.000000
9   apache02                             OSLinux-CPU_CPU_CPUWio            31    2.847925    9.983400    9.983400 2021-03-09 16:25:00+08:00                        2   3.505499
10      IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead            15  171.120000  570.400000  570.400000 2021-03-09 16:08:00+08:00                        1   3.333333
11   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps            15    0.015000    0.050000    0.050000 2021-03-09 16:25:00+08:00                        1   3.333333
12      IG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...            15    0.060000    0.200000    0.200000 2021-03-09 16:13:00+08:00                        1   3.333333
13   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead            15    0.240000    0.800000    0.800000 2021-03-09 16:25:00+08:00                        1   3.333333
14   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...            15    0.120000    0.400000    0.400000 2021-03-09 16:06:00+08:00                        1   3.333333
15      IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps            15    3.335000   11.116667   11.116667 2021-03-09 16:08:00+08:00                        1   3.333333
16  apache02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...            15   14.160000   47.200000   47.200000 2021-03-09 16:25:00+08:00                        1   3.333333
17  apache02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...            15    7.120000   22.800000   22.800000 2021-03-09 16:03:00+08:00                        1   3.202247
18   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...            15   20.800000   57.200000   57.200000 2021-03-09 16:27:00+08:00                        1   2.750000
19  apache02                         OSLinux-CPU_CPU_CPUCpuUtil            21    3.809400   10.349100   10.349100 2021-03-09 16:25:00+08:00                        1   2.716727```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_trace loaded earlier
# 1) Restrict to specified cmdb_ids
allowed_cmdbs = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
                 'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

df_t = df_trace[df_trace['cmdb_id'].isin(allowed_cmdbs)].copy()

# 2) Compute global stats over the ENTIRE trace series for each (cmdb_id, trace_name)
global_stats_trace = (
    df_t.groupby(['cmdb_id','trace_name'])['value']
    .agg(global_count='size',
         global_p95=lambda x: x.quantile(0.95),
         global_max='max')
    .reset_index()
)

# 3) Filter trace rows to the incident window (inclusive)
start_ts = 1615276800
end_ts = 1615278600
df_trace_window = df_t[(df_t['timestamp'] >= start_ts) & (df_t['timestamp'] <= end_ts)].copy()

# 4) For each series, compute window_max and find anomaly events where value > global_p95
window_max_df_trace = df_trace_window.groupby(['cmdb_id','trace_name'])['value'].agg(window_max='max').reset_index()

# Merge global stats into window rows to compare per-row for anomaly detection
df_tw_merged = df_trace_window.merge(global_stats_trace, on=['cmdb_id','trace_name'], how='left')

# Identify anomaly rows in window (value > global_p95)
df_tw_merged['is_anom'] = df_tw_merged['value'] > df_tw_merged['global_p95']

# Aggregate anomaly events: earliest timestamp and count within window for rows > p95
anom_events_trace = (
    df_tw_merged[df_tw_merged['is_anom']]
    .groupby(['cmdb_id','trace_name'])['timestamp']
    .agg(earliest_anomaly_timestamp='min', anomaly_count_in_window='size')
    .reset_index()
)

# Merge to form anomaly summary (only series with at least one anomaly)
anomaly_summary_trace = (
    anom_events_trace
    .merge(global_stats_trace, on=['cmdb_id','trace_name'], how='left')
    .merge(window_max_df_trace, on=['cmdb_id','trace_name'], how='left')
)

# Compute ratio = window_max / global_p95 safely (handle global_p95 == 0)
anomaly_summary_trace['ratio'] = np.where(
    anomaly_summary_trace['global_p95'] == 0,
    np.where(anomaly_summary_trace['window_max'] > 0, np.inf, 1.0),
    anomaly_summary_trace['window_max'] / anomaly_summary_trace['global_p95']
)

# Convert earliest anomaly timestamp to Asia/Shanghai tz-aware datetime
if not anomaly_summary_trace.empty:
    anomaly_summary_trace['earliest_anomaly_ts_local'] = pd.to_datetime(
        anomaly_summary_trace['earliest_anomaly_timestamp'], unit='s', utc=True
    ).dt.tz_convert(tz)

# Prepare final result, sorted by ratio desc, top 20
anomaly_summary_trace = anomaly_summary_trace.sort_values('ratio', ascending=False).reset_index(drop=True)
cols = ['cmdb_id','trace_name','global_p95','global_max','window_max',
        'earliest_anomaly_ts_local','anomaly_count_in_window','ratio']
result_df_trace = anomaly_summary_trace[cols].head(20)

# 5) If no series exceed global_p95, return top 10 series by window_max/global_p95 (even if <=1)
if result_df_trace.shape[0] == 0:
    # Build candidates from series that have window data
    candidates = window_max_df_trace.merge(global_stats_trace, on=['cmdb_id','trace_name'], how='left')
    candidates['ratio'] = np.where(
        candidates['global_p95'] == 0,
        np.where(candidates['window_max'] > 0, np.inf, 1.0),
        candidates['window_max'] / candidates['global_p95']
    )
    candidates = candidates.sort_values('ratio', ascending=False).reset_index(drop=True)
    # Select top 10 (no earliest timestamp/anomaly count available)
    result_df_trace = candidates[['cmdb_id','trace_name','global_p95','global_max','window_max','ratio']].head(10)

# Keep intermediate DataFrames in kernel for follow-up:
# global_stats_trace, df_trace_window, df_tw_merged, window_max_df_trace, anom_events_trace, anomaly_summary_trace

# Display compact result (max 20 rows)
result_df_trace

```
Out[3]:
```
summary = (
"Trace anomaly summary (window 2021-03-09 16:00–16:30 UTC+8):\n\n"
"Key findings:\n"
"- Strong, concentrated latency spikes occurred around 2021-03-09 16:24+08:00 across MG01 and MG02 traces and several docker-related spans.\n"
"- Top affected series (examples): MG02.trace.from_dockerB2.duration_mean (ratio 3.57), MG02.trace.to_dockerA1.duration_mean (3.48), MG02.trace.from_dockerB1.duration_mean (3.07), MG02.trace.from_dockerA2.duration_mean (2.86), MG01.trace.from_dockerA2.duration_mean (2.84), plus IG01 <-> Tomcat03 and Tomcat04 related traces (~2.84 and ~2.58).\n"
"- Each reported series has window_max > global_p95, with anomaly_count_in_window = 2 and earliest anomaly timestamp = 2021-03-09 16:24:00+08:00 in these top rows.\n\n"
"Interpretation and recommended next steps:\n"
"1) Investigate MG01/MG02 and their interactions with dockerA*/dockerB* services around 16:24 — these show the largest relative latency increases (2.4–3.6x over historical p95).\n"
"2) Correlate with network, CPU, and I/O metrics on MG and docker hosts and with application logs at 16:24 to find root causes (queueing, GC, network saturation, or slow downstream calls).\n"
"3) Check IG01 -> Tomcat03/04 paths as they also show ~2.6–2.8x spikes; investigate whether upstream MG/dockers caused downstream reverberation.\n\n"
"Summary prepared from trace global p95 comparisons and incident-window maxima; full intermediate tables are available in the kernel for follow-up."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name  global_p95  global_max  window_max earliest_anomaly_ts_local  anomaly_count_in_window     ratio
0       MG02  trace.from_dockerB2.duration_mean   10.539516   37.576455   37.576455 2021-03-09 16:24:00+08:00                        2  3.565292
1       MG02    trace.to_dockerA1.duration_mean    2.626645    9.147694    9.147694 2021-03-09 16:24:00+08:00                        2  3.482653
2       MG02  trace.from_dockerB1.duration_mean   11.636806   35.689477   35.689477 2021-03-09 16:24:00+08:00                        2  3.066948
3       MG02  trace.from_dockerA2.duration_mean   11.359609   32.517194   32.517194 2021-03-09 16:24:00+08:00                        2  2.862528
4       MG01  trace.from_dockerA2.duration_mean   11.957339   34.003059   34.003059 2021-03-09 16:24:00+08:00                        2  2.843698
5       IG01     trace.to_Tomcat03.duration_p95   13.004080   36.893250   36.893250 2021-03-09 16:24:00+08:00                        2  2.837052
6   Tomcat03       trace.from_IG01.duration_p95   13.004080   36.893250   36.893250 2021-03-09 16:24:00+08:00                        2  2.837052
7       MG01  trace.from_dockerB2.duration_mean   10.542751   29.802927   29.802927 2021-03-09 16:24:00+08:00                        2  2.826864
8       MG02  trace.from_dockerA1.duration_mean   12.421896   33.846931   33.846931 2021-03-09 16:24:00+08:00                        2  2.724780
9       MG01  trace.from_dockerB1.duration_mean   10.554103   27.775196   27.775196 2021-03-09 16:24:00+08:00                        2  2.631697
10      MG02     trace.to_dockerA1.duration_p95   17.594905   46.073850   46.073850 2021-03-09 16:24:00+08:00                        2  2.618590
11      MG01  trace.from_dockerA1.duration_mean   12.421709   32.448889   32.448889 2021-03-09 16:24:00+08:00                        2  2.612273
12      MG01     trace.to_dockerA2.duration_p95   18.867815   48.849150   48.849150 2021-03-09 16:24:00+08:00                        2  2.589020
13      MG01    trace.to_dockerA2.duration_mean    3.102419    8.027662    8.027662 2021-03-09 16:24:00+08:00                        2  2.587549
14  Tomcat04       trace.from_IG01.duration_p95   12.633960   32.624400   32.624400 2021-03-09 16:24:00+08:00                        2  2.582278
15      IG01     trace.to_Tomcat04.duration_p95   12.633960   32.624400   32.624400 2021-03-09 16:24:00+08:00                        2  2.582278
16      MG02    trace.to_dockerA2.duration_mean    3.372222    8.175312    8.175312 2021-03-09 16:24:00+08:00                        2  2.424310
17      MG01   trace.from_dockerA2.duration_p95   23.639565   56.734450   56.734450 2021-03-09 16:24:00+08:00                        2  2.399979
18      MG02   trace.from_dockerB1.duration_p95   25.117837   59.691100   59.691100 2021-03-09 16:24:00+08:00                        2  2.376443
19      MG01     trace.to_dockerA1.duration_p95   18.675097   43.926900   43.926900 2021-03-09 16:24:00+08:00                        2  2.352164```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_log and df_error from the kernel
allowed_cmdbs = ['apache02','apache01','MG01','MG02','IG01','Tomcat03','Tomcat04','Mysql01','Mysql02']
start_ts = 1615276800
end_ts = 1615278600

# 1) Filter both files to the window and allowed cmdbs
df_log_window = df_log[
    (df_log['timestamp'] >= start_ts) &
    (df_log['timestamp'] <= end_ts) &
    (df_log['cmdb_id'].isin(allowed_cmdbs))
].copy()

df_error_window = df_error[
    (df_error['timestamp'] >= start_ts) &
    (df_error['timestamp'] <= end_ts) &
    (df_error['cmdb_id'].isin(allowed_cmdbs))
].copy()

# Ensure tz-aware local timestamp column exists (use existing 'ts' if present; otherwise create)
if 'ts' not in df_log_window.columns and 'timestamp' in df_log_window.columns:
    df_log_window['ts'] = pd.to_datetime(df_log_window['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
if 'ts' not in df_error_window.columns and 'timestamp' in df_error_window.columns:
    df_error_window['ts'] = pd.to_datetime(df_error_window['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 2) From log.csv produce aggregate per cmdb_id
# Prepare base dataframe with all allowed cmdbs to show NA where absent
agg_base = pd.DataFrame({'cmdb_id': allowed_cmdbs})

# total_rows_in_window
total_rows = df_log_window.groupby('cmdb_id').size().rename('total_rows_in_window').reset_index()

# sum_log_error_count (where log_name == 'log.error_count')
err_rows = df_log_window[df_log_window['log_name'] == 'log.error_count'].copy()
sum_log_error = err_rows.groupby('cmdb_id')['value'].sum().rename('sum_log_error_count').reset_index()

# earliest_error_timestamp_local (smallest ts where log_name=='log.error_count' and value>0)
err_pos = err_rows[err_rows['value'] > 0].copy()
earliest_err = (
    err_pos.groupby('cmdb_id')['timestamp']
    .min()
    .rename('earliest_error_timestamp')
    .reset_index()
)
# convert to tz-aware local string
if not earliest_err.empty:
    earliest_err['earliest_error_timestamp_local'] = pd.to_datetime(
        earliest_err['earliest_error_timestamp'], unit='s', utc=True
    ).dt.tz_convert(tz)
    earliest_err = earliest_err[['cmdb_id','earliest_error_timestamp_local']]
else:
    earliest_err = pd.DataFrame(columns=['cmdb_id','earliest_error_timestamp_local'])

# max_log_gc_pause (log_name contains 'gc_pause')
gc_rows = df_log_window[df_log_window['log_name'].str.contains('gc_pause', na=False)].copy()
max_gc = gc_rows.groupby('cmdb_id')['value'].max().rename('max_log_gc_pause').reset_index()

# Example rows: log_name == 'log.error_count' or contains 'gc_pause' or contains 'row_count'
candidates = df_log_window[
    (df_log_window['log_name'] == 'log.error_count') |
    (df_log_window['log_name'].str.contains('gc_pause', na=False)) |
    (df_log_window['log_name'].str.contains('row_count', na=False))
].copy()

# sort by value desc and take top 20 globally, then up to 5 per cmdb
candidates_sorted = candidates.sort_values('value', ascending=False).head(20)
# convert ts to local string for examples
if not candidates_sorted.empty:
    candidates_sorted['ts_local'] = pd.to_datetime(candidates_sorted['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    candidates_sorted['example_str'] = candidates_sorted.apply(
        lambda r: f"{r['ts_local'].strftime('%Y-%m-%d %H:%M:%S%z')}|{r['log_name']}|{r['value']}", axis=1
    )
else:
    candidates_sorted['example_str'] = []

# take up to 5 examples per cmdb from the global top20
examples_per_cmdb = (
    candidates_sorted.groupby('cmdb_id').head(5)[['cmdb_id','example_str']]
    .groupby('cmdb_id')['example_str'].apply(lambda s: '; '.join(s.tolist())).reset_index()
)

# Merge all aggregates into one compact table
agg_table = agg_base.merge(total_rows, on='cmdb_id', how='left') \
    .merge(sum_log_error, on='cmdb_id', how='left') \
    .merge(earliest_err, on='cmdb_id', how='left') \
    .merge(max_gc, on='cmdb_id', how='left') \
    .merge(examples_per_cmdb, on='cmdb_id', how='left')

# Replace missing values with NA and keep columns compact
agg_table['total_rows_in_window'] = agg_table['total_rows_in_window'].fillna(0).astype(int)
agg_table['sum_log_error_count'] = agg_table['sum_log_error_count'].fillna(0)
agg_table['earliest_error_timestamp_local'] = agg_table['earliest_error_timestamp_local'].fillna('NA')
agg_table['max_log_gc_pause'] = agg_table['max_log_gc_pause'].where(~agg_table['max_log_gc_pause'].isna(), 'NA')
agg_table['example_rows'] = agg_table['example_str'].fillna('NA')
agg_table = agg_table[['cmdb_id','total_rows_in_window','sum_log_error_count',
                       'earliest_error_timestamp_local','max_log_gc_pause','example_rows']]

# 3) From error_logs.csv list all rows in the window (timestamp_local, cmdb_id, message) up to 20 rows
if not df_error_window.empty:
    df_error_window['timestamp_local'] = pd.to_datetime(df_error_window['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    error_rows_display = df_error_window[['timestamp_local','cmdb_id','message']].sort_values('timestamp_local').head(20).reset_index(drop=True)
else:
    error_rows_display = pd.DataFrame(columns=['timestamp_local','cmdb_id','message'])

# Final compact outputs (max 20 rows each)
agg_table, error_rows_display.head(20)

```
Out[4]:
```
```python
# Prepare a concise plain-English summary string from the already-computed results
# Uses agg_table and error_rows_display variables present in the kernel
lines = []
lines.append("Log & error summary for window 2021-03-09 16:00–16:30 (UTC+8):")
lines.append("")

lines.append("Aggregated log.csv by cmdb_id (fields: total_rows, sum_log_error_count, earliest_error_ts, max_log_gc_pause, example_rows[truncated]):")
for _, r in agg_table.iterrows():
    cmdb = r['cmdb_id']
    total = r['total_rows_in_window']
    sum_err = r['sum_log_error_count']
    earliest = r['earliest_error_timestamp_local']
    max_gc = r['max_log_gc_pause']
    examples = r['example_rows']
    if isinstance(examples, str) and len(examples) > 120:
        examples_short = examples[:117] + '...'
    else:
        examples_short = examples
    lines.append(f"- {cmdb}: total_rows={total}, sum_log_error_count={sum_err}, earliest_error_ts={earliest}, max_log_gc_pause={max_gc}, examples={examples_short}")

lines.append("")
lines.append("Notable observations:")
lines.append("- apache02: 62 log rows in-window with sum_log_error_count=1 and earliest error at 2021-03-09 16:24:00+08:00. Example rows include high row_count values.")
lines.append("- apache01: 62 rows, no error_count logged in-window; example row_count entries present.")
lines.append("- Tomcat03/Tomcat04: many log rows (84/83) and small gc_pause maxima (0.11 / 0.12) recorded in-window.")
lines.append("- MG01, MG02, IG01: no log rows in the window (total_rows=0) per the filtered log.csv.")
lines.append("- Mysql01 and Mysql02: some log rows (60 and 10) but no log.error_count >0 recorded in-window.")

lines.append("")
lines.append("error_logs rows in-window (timestamp_local, cmdb_id, message) up to 20 rows:")
if error_rows_display.shape[0] == 0:
    lines.append("- none")
else:
    for _, er in error_rows_display.iterrows():
        ts = er['timestamp_local']
        cmdb = er['cmdb_id']
        msg = er['message']
        # truncate message for compactness
        msg_short = (msg[:140] + '...') if isinstance(msg, str) and len(msg) > 140 else msg
        lines.append(f"- {ts} | {cmdb} | {msg_short}")

lines.append("")
lines.append("Suggested next steps:")
lines.append("- Investigate apache02 around 16:24 (log.error_count present and an error_logs entry at 16:24:06).")
lines.append("- Review high row_count examples for apache01/apache02 and GC pause entries for Tomcat03/04 if correlated with latency.")
lines.append("- Check why MG01/MG02/IG01 have no log rows in this window (data gap or different logging cadence).")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  total_rows_in_window  sum_log_error_count earliest_error_timestamp_local max_log_gc_pause                                       example_rows
0  apache02                    62                  1.0      2021-03-09 16:24:00+08:00               NA  2021-03-09 16:02:00+0800|log.row_count|1040.0;...
1  apache01                    62                  0.0                             NA               NA  2021-03-09 16:02:00+0800|log.row_count|1018.0;...
2      MG01                     0                  0.0                             NA               NA                                                 NA
3      MG02                     0                  0.0                             NA               NA                                                 NA
4      IG01                     0                  0.0                             NA               NA                                                 NA
5  Tomcat03                    84                  0.0                             NA             0.11                                                 NA
6  Tomcat04                    83                  0.0                             NA             0.12                                                 NA
7   Mysql01                    60                  0.0                             NA               NA                                                 NA
8   Mysql02                    10                  0.0                             NA               NA                                                 NA,             timestamp_local   cmdb_id                                            message
0 2021-03-09 16:24:06+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest3.json...)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse intermediate kernel variables:
# anomaly_summary (metric), anomaly_summary_trace (trace), agg_table (logs)
# They should exist from prior steps.

# 1) Candidates: union of cmdb_ids present in the three tables
metric_cmdbs = set(anomaly_summary['cmdb_id']) if 'anomaly_summary' in globals() and not anomaly_summary.empty else set()
trace_cmdbs = set(anomaly_summary_trace['cmdb_id']) if 'anomaly_summary_trace' in globals() and not anomaly_summary_trace.empty else set()
log_cmdbs = set(agg_table['cmdb_id']) if 'agg_table' in globals() and not agg_table.empty else set()

candidates = sorted(metric_cmdbs.union(trace_cmdbs).union(log_cmdbs))
cand_df = pd.DataFrame({'cmdb_id': candidates})

# Helper to extract per-cmdb earliest timestamp and max ratio from metric anomalies
if 'anomaly_summary' in globals() and not anomaly_summary.empty:
    m = anomaly_summary.copy()
    # Ensure earliest_anomaly_ts_local is datetime tz-aware (may already be)
    if 'earliest_anomaly_ts_local' in m.columns:
        m['metric_ts'] = pd.to_datetime(m['earliest_anomaly_ts_local'], utc=True, errors='coerce').dt.tz_convert(tz)
    else:
        m['metric_ts'] = pd.NaT
    m['metric_ratio'] = pd.to_numeric(m['ratio'], errors='coerce').fillna(0)
    metric_agg = m.groupby('cmdb_id').agg(
        earliest_metric_anomaly_ts_local=('metric_ts', 'min'),
        max_metric_ratio=('metric_ratio', 'max')
    ).reset_index()
else:
    metric_agg = pd.DataFrame(columns=['cmdb_id','earliest_metric_anomaly_ts_local','max_metric_ratio'])

# Trace aggregates
if 'anomaly_summary_trace' in globals() and not anomaly_summary_trace.empty:
    t = anomaly_summary_trace.copy()
    if 'earliest_anomaly_ts_local' in t.columns:
        t['trace_ts'] = pd.to_datetime(t['earliest_anomaly_ts_local'], utc=True, errors='coerce').dt.tz_convert(tz)
    else:
        t['trace_ts'] = pd.NaT
    t['trace_ratio'] = pd.to_numeric(t['ratio'], errors='coerce').fillna(0)
    trace_agg = t.groupby('cmdb_id').agg(
        earliest_trace_anomaly_ts_local=('trace_ts', 'min'),
        max_trace_ratio=('trace_ratio', 'max')
    ).reset_index()
else:
    trace_agg = pd.DataFrame(columns=['cmdb_id','earliest_trace_anomaly_ts_local','max_trace_ratio'])

# Log aggregates: use agg_table produced earlier
if 'agg_table' in globals() and not agg_table.empty:
    l = agg_table.copy()
    # earliest_error_timestamp_local may be string 'NA' or Timestamp; coerce to datetime
    l['earliest_log_error_ts'] = pd.to_datetime(l['earliest_error_timestamp_local'], utc=True, errors='coerce').dt.tz_convert(tz)
    # has_error_log boolean
    l['has_error_log'] = ~l['earliest_log_error_ts'].isna()
    log_agg = l[['cmdb_id','earliest_log_error_ts','has_error_log']].rename(
        columns={'earliest_log_error_ts':'earliest_log_error_ts_local'}
    )
else:
    log_agg = pd.DataFrame(columns=['cmdb_id','earliest_log_error_ts_local','has_error_log'])

# Merge aggregates into candidate table
df = cand_df.merge(metric_agg, on='cmdb_id', how='left') \
            .merge(trace_agg, on='cmdb_id', how='left') \
            .merge(log_agg, on='cmdb_id', how='left')

# Fill NA ratios with 0 for comparison, but keep NaT for timestamps
df['max_metric_ratio'] = df['max_metric_ratio'].fillna(0)
df['max_trace_ratio'] = df['max_trace_ratio'].fillna(0)

# 2) earliest_overall_ts_local: minimum of the three timestamp columns (ignoring NaT)
ts_cols = ['earliest_metric_anomaly_ts_local','earliest_trace_anomaly_ts_local','earliest_log_error_ts_local']
# ensure columns exist
for c in ts_cols:
    if c not in df.columns:
        df[c] = pd.NaT

# Calculate earliest overall timestamp per row
df['earliest_overall_ts'] = df[ts_cols].min(axis=1)

# Determine primary_source
def choose_primary(row):
    overall = row['earliest_overall_ts']
    if pd.isna(overall):
        return 'NA'
    candidates_src = []
    # Check which sources match the overall timestamp (use equality of Timestamps)
    for src, col in [('metric','earliest_metric_anomaly_ts_local'),
                     ('trace','earliest_trace_anomaly_ts_local'),
                     ('log','earliest_log_error_ts_local')]:
        val = row[col]
        if not pd.isna(val) and val == overall:
            candidates_src.append(src)
    if len(candidates_src) == 1:
        return candidates_src[0]
    # If multiple tied, pick source with larger ratio (log has no ratio -> treat as 0)
    ratios = {'metric': row.get('max_metric_ratio',0) or 0,
              'trace': row.get('max_trace_ratio',0) or 0,
              'log': 0}
    # restrict to candidates
    cand_ratios = {s: ratios[s] for s in candidates_src}
    if cand_ratios:
        max_ratio = max(cand_ratios.values())
        # sources with max_ratio among candidates
        top_sources = [s for s,r in cand_ratios.items() if r == max_ratio]
        # tie-break preference: log > trace > metric
        for pref in ['log','trace','metric']:
            if pref in top_sources:
                return pref
        return top_sources[0]
    # If no candidate matched exactly (shouldn't happen), pick source with earliest non-NA timestamp by preference order
    for pref in ['log','trace','metric']:
        col = {'log':'earliest_log_error_ts_local','trace':'earliest_trace_anomaly_ts_local','metric':'earliest_metric_anomaly_ts_local'}[pref]
        if not pd.isna(row[col]):
            return pref
    return 'NA'

df['primary_source'] = df.apply(choose_primary, axis=1)

# overall_max_ratio = max of metric and trace ratios
df['overall_max_ratio'] = df[['max_metric_ratio','max_trace_ratio']].max(axis=1)

# Format timestamp columns as localized strings or 'NA'
def fmt_ts(x):
    if pd.isna(x):
        return 'NA'
    # ensure tz Asia/Shanghai
    ts = pd.to_datetime(x, utc=True, errors='coerce')
    if pd.isna(ts):
        return 'NA'
    ts = ts.dt.tz_convert(tz) if isinstance(ts, pd.Series) else ts.tz_convert(tz)
    # single Timestamp case:
    if isinstance(ts, pd.Series):
        return ts.apply(lambda t: t.strftime('%Y-%m-%d %H:%M:%S%z') if not pd.isna(t) else 'NA')
    else:
        return ts.strftime('%Y-%m-%d %H:%M:%S%z')

# Apply formatting for display columns
df['earliest_metric_anomaly_ts_local'] = df['earliest_metric_anomaly_ts_local'].apply(lambda x: x.strftime('%Y-%m-%d %H:%M:%S%z') if not pd.isna(x) else 'NA')
df['earliest_trace_anomaly_ts_local']  = df['earliest_trace_anomaly_ts_local'].apply(lambda x: x.strftime('%Y-%m-%d %H:%M:%S%z') if not pd.isna(x) else 'NA')
df['earliest_log_error_ts_local']     = df['earliest_log_error_ts_local'].apply(lambda x: x.strftime('%Y-%m-%d %H:%M:%S%z') if not pd.isna(x) else 'NA')
df['earliest_overall_ts_local']       = df['earliest_overall_ts'].apply(lambda x: x.strftime('%Y-%m-%d %H:%M:%S%z') if not pd.isna(x) else 'NA')

# Select and order columns for compact output
out_cols = [
    'cmdb_id',
    'earliest_metric_anomaly_ts_local',
    'max_metric_ratio',
    'earliest_trace_anomaly_ts_local',
    'max_trace_ratio',
    'earliest_log_error_ts_local',
    'has_error_log',
    'earliest_overall_ts_local',
    'primary_source',
    'overall_max_ratio'
]

out_df = df[out_cols].copy()

# Sort: earliest_overall_ts_local ascending (NA last), then overall_max_ratio descending
# For sorting, convert earliest_overall_ts_local back to datetime for proper ordering
sort_ts = pd.to_datetime(df['earliest_overall_ts'], utc=True, errors='coerce')
out_df['_sort_ts'] = sort_ts
out_df = out_df.sort_values(by=['_sort_ts','overall_max_ratio'], ascending=[True, False], na_position='last').drop(columns=['_sort_ts'])

# Limit to top 20 rows
out_df = out_df.head(20).reset_index(drop=True)

# Display compact candidate timeline table
out_df

```
Out[5]:
```
Timeline candidate summary (compact)

Window: 2021-03-09 16:00–16:30 (UTC+8). Candidates are union of metric, trace and log signals.

Key timeline & severity highlights (one line per candidate, earliest timestamp — primary source — peak ratio):

- Mysql01 — earliest 2021-03-09 16:00 — primary: metric — overall_max_ratio: inf (metric spike, earliest metric anomaly 16:00).
- Mysql02 — earliest 2021-03-09 16:00 — primary: metric — overall_max_ratio: inf (multiple MySQL metric spikes at 16:00/16:26).
- Tomcat04 — earliest 2021-03-09 16:00 — primary: metric — overall_max_ratio: 20.0 (large metric ratio); trace also shows ~2.58 at same time.
- apache01 — earliest 2021-03-09 16:00 — primary: metric — overall_max_ratio: 4.0 (CPU/disk metric spike).
- IG02 — earliest 2021-03-09 16:00 — primary: metric — overall_max_ratio: 3.333 (disk I/O spike; trace ratio ~2.22).
- Tomcat03 — earliest overall 2021-03-09 16:00 — primary: trace — overall_max_ratio: 2.837 (trace latency spike; metric ratio ~1.82 at ~16:01).
- Tomcat01 — earliest overall 2021-03-09 16:00–16:01 — primary: trace — overall_max_ratio: 2.216 (trace > metric).
- Tomcat02 — earliest 2021-03-09 16:00 — primary: metric — overall_max_ratio: 1.848.
- Redis01 — earliest 2021-03-09 16:00 — primary: metric — overall_max_ratio: 1.538.
- Redis02 — earliest 2021-03-09 16:00 — primary: metric — overall_max_ratio: 1.535.
- apache02 — earliest 2021-03-09 16:01 — primary: metric — overall_max_ratio: inf; has_error_log=True with log/error at 2021-03-09 16:24 (log+error entry).
- MG02 — earliest 2021-03-09 16:01 — primary: metric — overall_max_ratio: 10.0 (metric spike) and strong trace spikes at ~16:24 (trace ratio ~3.57).
- MG01 — earliest 2021-03-09 16:02 — primary: metric — overall_max_ratio: 10.0 (metric spike) and trace ratio ~2.84 at ~16:24.
- IG01 — earliest 2021-03-09 16:02 — primary: metric — overall_max_ratio: 3.333; trace also shows ~2.84 at 16:02.

Interpretation and recommended next steps (concise):

1. Prioritize investigation order by severity and evidence:
   - Highest priority: Mysql01, Mysql02, apache02 (inf ratios), Tomcat04 (ratio 20), MG01/MG02 (ratio 10).
2. Note timing pattern:
   - Metric anomalies begin at ~16:00–16:02 on many hosts (disk/CPU/socket counts).
   - Trace latency spikes concentrate at ~16:24 (MG and docker-related traces) and affect downstream services (IG <-> Tomcat).
   - apache02 has both metric anomalies and a log/error entry at ~16:24 — strong choke point candidate.
3. Correlate sources around these times:
   - For 16:00–16:02: inspect metrics (disk I/O, TCP CLOSE/FIN counts, CPU WIO) on Mysql, Tomcat04, apache01/02, IG nodes.
   - For ~16:24: inspect MG01/MG02 and dockerA*/dockerB* interactions (trace latency), and check apache02 logs at 16:24.
4. Use logs, traces, and metrics together to identify root cause (e.g., socket backlog/CLOSE-WAIT, disk I/O saturation, downstream latency amplification).

Summary: The incident shows two concentrated phases — early metric spikes (~16:00–16:02) across DB/web/IG nodes and later distributed trace latency spikes around 16:24 centered on MG/dockers — with apache02 and MySQL instances among top candidates to investigate first.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id earliest_metric_anomaly_ts_local  max_metric_ratio earliest_trace_anomaly_ts_local  max_trace_ratio earliest_log_error_ts_local has_error_log earliest_overall_ts_local primary_source  overall_max_ratio
0    Mysql01         2021-03-09 16:00:00+0800               inf                              NA         0.000000                          NA         False  2021-03-09 16:00:00+0800         metric                inf
1    Mysql02         2021-03-09 16:00:00+0800               inf                              NA         0.000000                          NA         False  2021-03-09 16:00:00+0800         metric                inf
2   Tomcat04         2021-03-09 16:00:00+0800         20.000000        2021-03-09 16:00:00+0800         2.582278                          NA         False  2021-03-09 16:00:00+0800         metric          20.000000
3   apache01         2021-03-09 16:00:00+0800          4.000000                              NA         0.000000                          NA         False  2021-03-09 16:00:00+0800         metric           4.000000
4       IG02         2021-03-09 16:00:00+0800          3.333333        2021-03-09 16:00:00+0800         2.216390                          NA           NaN  2021-03-09 16:00:00+0800         metric           3.333333
5   Tomcat03         2021-03-09 16:01:00+0800          1.818182        2021-03-09 16:00:00+0800         2.837052                          NA         False  2021-03-09 16:00:00+0800          trace           2.837052
6   Tomcat01         2021-03-09 16:01:00+0800          1.818182        2021-03-09 16:00:00+0800         2.216390                          NA           NaN  2021-03-09 16:00:00+0800          trace           2.216390
7   Tomcat02         2021-03-09 16:00:00+0800          1.818182        2021-03-09 16:02:00+0800         1.848441                          NA           NaN  2021-03-09 16:00:00+0800         metric           1.848441
8    Redis01         2021-03-09 16:00:00+0800          1.538462                              NA         0.000000                          NA           NaN  2021-03-09 16:00:00+0800         metric           1.538462
9    Redis02         2021-03-09 16:00:00+0800          1.535128                              NA         0.000000                          NA           NaN  2021-03-09 16:00:00+0800         metric           1.535128
10  apache02         2021-03-09 16:01:00+0800               inf                              NA         0.000000    2021-03-09 16:24:00+0800          True  2021-03-09 16:01:00+0800         metric                inf
11      MG02         2021-03-09 16:01:00+0800         10.000000        2021-03-09 16:02:00+0800         3.565292                          NA         False  2021-03-09 16:01:00+0800         metric          10.000000
12      MG01         2021-03-09 16:02:00+0800         10.000000        2021-03-09 16:02:00+0800         2.843698                          NA         False  2021-03-09 16:02:00+0800         metric          10.000000
13      IG01         2021-03-09 16:02:00+0800          3.333333        2021-03-09 16:02:00+0800         2.837052                          NA         False  2021-03-09 16:02:00+0800         metric           3.333333```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# reuse df_trace from kernel and ensure tz
tz = pytz.timezone('Asia/Shanghai')

# filter parameters
target_ts = 1615278240  # 2021-03-09 16:24:00 UTC+8
allowed_cmdbs = ['MG01','MG02','IG01','Tomcat03','Tomcat04','apache02','Mysql01','Mysql02']

# 1) Extract rows for the timestamp and cmdb_ids
df_trace_at_ts = df_trace[(df_trace['timestamp'] == target_ts) & (df_trace['cmdb_id'].isin(allowed_cmdbs))].copy()

# add localized timestamp for clarity (not required in final cols but useful if needed)
if not df_trace_at_ts.empty:
    df_trace_at_ts['ts_local'] = pd.to_datetime(df_trace_at_ts['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# compute count of rows per cmdb_id (within this filtered slice)
counts_per_cmdb = df_trace_at_ts.groupby('cmdb_id').size().reset_index(name='count')

# prepare compact table: cmdb_id, trace_name, value, count (merge counts)
df_display = df_trace_at_ts.merge(counts_per_cmdb, on='cmdb_id', how='left')
df_display = df_display[['cmdb_id','trace_name','value','count']].sort_values(['cmdb_id','value'], ascending=[True, False]).head(50).reset_index(drop=True)

# 2) Top 20 rows where trace_name contains 'to_' or 'from_' sorted by value desc
mask_dir = df_trace_at_ts['trace_name'].str.contains(r'\bto_|from_', na=False)
top20_calls = df_trace_at_ts[mask_dir].sort_values('value', ascending=False).head(20)[['cmdb_id','trace_name','value']].reset_index(drop=True)

# Display compact results: the per-row table (max 50 rows) and the top-20 caller->callee directions
df_display, top20_calls

```
Out[6]:
```
Snapshot at 2021-03-09 16:24:00 (UTC+8)

Summary of extracted traces (selected hosts: MG01, MG02, IG01, Tomcat03, Tomcat04, apache02, Mysql01, Mysql02):

- Row-count examples per host (from the extracted slice): IG01 had 15 trace rows (e.g., trace.self.row_count = 1346); MG01 had 39 trace rows in this slice. (Full per-host counts are included in the extracted table.)

Top directional (caller→callee) traces at 16:24 — highest row_count / latency first (top 20):
1. Tomcat04 — trace.from_IG02.row_count = 191.0  
2. MG02 — trace.from_Tomcat02.row_count = 190.0  
3. MG02 — trace.to_dockerB2.row_count = 182.0  
4. MG01 — trace.from_Tomcat04.row_count = 181.0  
5. Tomcat04 — trace.to_MG01.row_count = 181.0  
6. MG01 — trace.from_Tomcat02.row_count = 181.0  
7. MG02 — trace.from_Tomcat04.row_count = 180.0  
8. Tomcat04 — trace.to_MG02.row_count = 180.0  
9. MG01 — trace.from_Tomcat01.row_count = 179.0  
10. IG01 — trace.to_Tomcat01.row_count = 176.0  
11. MG01 — trace.to_dockerB1.row_count = 175.0  
12. MG02 — trace.to_dockerB1.row_count = 174.0  
13. MG01 — trace.to_dockerB2.row_count = 173.0  
14. IG01 — trace.to_Tomcat02.row_count = 171.0  
15. Tomcat03 — trace.from_IG02.row_count = 166.0  
16. IG01 — trace.to_Tomcat04.row_count = 165.0  
17. Tomcat04 — trace.from_IG01.row_count = 165.0  
18. MG02 — trace.from_Tomcat01.row_count = 162.0  
19. MG02 — trace.from_Tomcat03.row_count = 156.0  
20. Tomcat03 — trace.to_MG02.row_count = 156.0

Key takeaways:
- Heavy call/row_count activity is concentrated around MG↔Tomcat, IG→Tomcat, and MG→dockerB interactions at 16:24.  
- IG01 shows a very large self.row_count (1346) and many outgoing calls to Tomcat nodes (e.g., 176,171,165,136), indicating high frontend throughput at that moment.  
- MG nodes show many high-count directional calls to docker and Tomcat services — MG01/MG02 are central to the high traffic/latency pattern.  
- These directional top entries point to where to investigate latency/queueing (MG <-> docker, IG -> Tomcat, Tomcat <-> MG paths).

If you want, I can (a) print the compact 50-row table with counts per cmdb_id, or (b) show more details for a specific host or trace_name.

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                         trace_name        value  count
0     IG01               trace.self.row_count  1346.000000     15
1     IG01        trace.to_Tomcat01.row_count   176.000000     15
2     IG01        trace.to_Tomcat02.row_count   171.000000     15
3     IG01        trace.to_Tomcat04.row_count   165.000000     15
4     IG01        trace.to_Tomcat03.row_count   136.000000     15
..     ...                                ...          ...    ...
45    MG01           trace.self.duration_mean    13.607927     39
46    MG01  trace.from_Tomcat02.duration_mean    10.537376     39
47    MG01  trace.from_Tomcat04.duration_mean     9.788735     39
48    MG01  trace.from_Tomcat03.duration_mean     9.614268     39
49    MG01  trace.from_Tomcat01.duration_mean     8.885844     39

[50 rows x 4 columns],      cmdb_id                     trace_name  value
0   Tomcat04      trace.from_IG02.row_count  191.0
1       MG02  trace.from_Tomcat02.row_count  190.0
2       MG02    trace.to_dockerB2.row_count  182.0
3       MG01  trace.from_Tomcat04.row_count  181.0
4   Tomcat04        trace.to_MG01.row_count  181.0
5       MG01  trace.from_Tomcat02.row_count  181.0
6       MG02  trace.from_Tomcat04.row_count  180.0
7   Tomcat04        trace.to_MG02.row_count  180.0
8       MG01  trace.from_Tomcat01.row_count  179.0
9       IG01    trace.to_Tomcat01.row_count  176.0
10      MG01    trace.to_dockerB1.row_count  175.0
11      MG02    trace.to_dockerB1.row_count  174.0
12      MG01    trace.to_dockerB2.row_count  173.0
13      IG01    trace.to_Tomcat02.row_count  171.0
14  Tomcat03      trace.from_IG02.row_count  166.0
15      IG01    trace.to_Tomcat04.row_count  165.0
16  Tomcat04      trace.from_IG01.row_count  165.0
17      MG02  trace.from_Tomcat01.row_count  162.0
18      MG02  trace.from_Tomcat03.row_count  156.0
19  Tomcat03        trace.to_MG02.row_count  156.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse df_metric from kernel
allowed_cmdbs = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
                 'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']
target_ts = 1615278240  # 2021-03-09 16:24:00 UTC+8

# 1) Compute global stats for all (cmdb_id, kpi_name) using the ENTIRE metric.csv series (rule 9)
global_stats_full = (
    df_metric.groupby(['cmdb_id','kpi_name'])['value']
    .agg(global_count='size',
         global_p95=lambda x: x.quantile(0.95),
         global_max='max')
    .reset_index()
)

# 2) Extract rows at the target timestamp and allowed cmdbs
df_at_ts = df_metric[
    (df_metric['timestamp'] == target_ts) &
    (df_metric['cmdb_id'].isin(allowed_cmdbs))
].copy()

# Merge global stats into the target-time rows
df_merged = df_at_ts.merge(global_stats_full, on=['cmdb_id','kpi_name'], how='left')

# 3) Compute ratio, handling global_p95 == 0 (show as np.inf)
df_merged['value_at_16_24'] = df_merged['value']
df_merged['global_p95'] = pd.to_numeric(df_merged['global_p95'], errors='coerce').fillna(0)
df_merged['global_max'] = pd.to_numeric(df_merged['global_max'], errors='coerce').fillna(0)

df_merged['ratio'] = np.where(
    df_merged['global_p95'] == 0,
    np.where(df_merged['value_at_16_24'] > 0, np.inf, 1.0),
    df_merged['value_at_16_24'] / df_merged['global_p95']
)

# 4) Select series where value_at_16_24 > global_p95
breaches = df_merged[df_merged['value_at_16_24'] > df_merged['global_p95']].copy()

if breaches.empty:
    # If no breaches, return top 20 by ratio (even if <=1)
    candidates = df_merged.copy()
    result_df = (
        candidates[['cmdb_id','kpi_name','value_at_16_24','global_p95','global_max','ratio']]
        .sort_values('ratio', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    result_df = (
        breaches[['cmdb_id','kpi_name','value_at_16_24','global_p95','global_max','ratio']]
        .sort_values('ratio', ascending=False)
        .head(50)
        .reset_index(drop=True)
    )

# 5) Small summary: counts per cmdb_id of breached KPIs at 16:24
breach_counts = breaches.groupby('cmdb_id').size().reset_index(name='breached_kpi_count').sort_values('breached_kpi_count', ascending=False)

# Keep outputs compact
result_df, breach_counts.head(50)

```
Out[7]:
```
Snapshot at 2021-03-09 16:24:00 (UTC+8) — metric breaches vs global p95

What I checked
- For each (cmdb_id, kpi_name) present at 16:24, I compared value_at_16_24 to the global 95th percentile computed over the entire metric.csv series for that series (global_p95, global_max).
- Reported only series with value_at_16_24 > global_p95, sorted by ratio = value_at_16_24 / global_p95 (top 50; 33 breaches found).

Top anomalies (highlights)
- apache02 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT: value=1, global_p95=0 → ratio = inf (socket CLOSE-WAIT observed).
- Mysql02 — heavy InnoDB activity:
  - InnoDB pages written / dblwr pages written: ratio ≈ 2.54
  - InnoDB data written: ratio ≈ 2.34 (very large absolute value)
- MG02 — OSLinux NETWORK TCP-FIN-WAIT: ratio ≈ 2.31 (elevated FIN-WAIT socket counts)
- Several other disk / I/O / network KPIs on MySQL, IG, MG, Tomcat and Apache nodes exceeded historic p95 by 2–3x.
- Many ratios near or just above 1.0 for less-severe breaches (small relative excursions).

Counts of breached KPIs per host (how many KPIs at 16:24 exceeded their global p95)
- Tomcat02: 7
- Mysql02: 6
- Tomcat03: 4
- Mysql01: 3
- MG02: 2
- apache02: 2
- IG02: 2
- Redis01: 2
- IG01: 1
- Tomcat01: 1
- Redis02: 1
- Tomcat04: 1
- apache01: 1

Interpretation & immediate suggestions
- Priority items: apache02 (TCP CLOSE-WAIT), Mysql02 (large InnoDB writes and related I/O metrics), MG02 (socket FIN-WAIT), and hosts with multiple breached KPIs (Tomcat02, Tomcat03).
- These indicate socket lifecycle issues and elevated disk/network I/O around 16:24 — investigate connections (CLOSE/FIN wait), disk throughput, and MySQL write bursts.
- Correlate with trace spikes (~16:24), logs (apache02 error at ~16:24), and other metric spikes to find root cause (e.g., downstream slowdowns causing queues and connection accumulation).

If you want, I can:
- Print the full compact table of all 33 breached series (cmdb_id, kpi_name, value_at_16_24, global_p95, global_max, ratio), or
- Focus on detailed metrics for a specific host (e.g., Mysql02 or apache02).

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                        kpi_name  value_at_16_24    global_p95    global_max     ratio
0   apache02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    1.000000e+00  0.000000e+00  1.000000e+00       inf
1    Mysql02           Mysql-MySQL_3306_Innodb pages written    6.906670e+01  2.718330e+01  6.906670e+01  2.540777
2    Mysql02     Mysql-MySQL_3306_Innodb dblwr pages written    6.906670e+01  2.718330e+01  6.906670e+01  2.540777
3    Mysql02            Mysql-MySQL_3306_Innodb data written    2.423765e+06  1.037397e+06  2.423765e+06  2.336391
4       MG02    OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    3.000000e+00  1.300000e+00  3.000000e+00  2.307692
..       ...                                             ...             ...           ...           ...       ...
28   Mysql01     OSLinux-NTP_197.30.1.68_NtpServerTimeOffset    1.410000e-01  1.408000e-01  1.410000e-01  1.001420
29   Redis01    OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut    5.132368e+09  5.131838e+09  5.132368e+09  1.000103
30   Redis01     OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn    5.657567e+09  5.657005e+09  5.657567e+09  1.000099
31  apache01         OSLinux-OSLinux_FILESYSTEM_-_FSCapacity    4.951614e+01  4.951611e+01  4.951614e+01  1.000001
32   Mysql01                 Mysql-MySQL_3306_Bytes Received   -9.406350e+03 -9.427672e+03 -9.406350e+03  0.997738

[33 rows x 6 columns],      cmdb_id  breached_kpi_count
8   Tomcat02                   7
4    Mysql02                   6
9   Tomcat03                   4
3    Mysql01                   3
2       MG02                   2
12  apache02                   2
1       IG02                   2
5    Redis01                   2
0       IG01                   1
7   Tomcat01                   1
6    Redis02                   1
10  Tomcat04                   1
11  apache01                   1)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[8]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz
import re

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables if present, otherwise load files
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Parameters
allowed_cmdbs = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
                 'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']
target_ts = 1615278240  # 2021-03-09 16:24:00 UTC+8

# --- Step 1-2: metric breaches at target timestamp ---
# Compute global stats for each (cmdb_id, kpi_name) using entire metric series
global_stats_full = (
    df_metric.groupby(['cmdb_id','kpi_name'])['value']
    .agg(global_count='size',
         global_p95=lambda x: x.quantile(0.95),
         global_max='max')
    .reset_index()
)

# Rows at target timestamp for allowed components, merge global stats
df_metric_at = df_metric[
    (df_metric['timestamp'] == target_ts) &
    (df_metric['cmdb_id'].isin(allowed_cmdbs))
].merge(global_stats_full, on=['cmdb_id','kpi_name'], how='left')

# Compute ratio and breach flag
df_metric_at['value_at_16_24'] = df_metric_at['value']
df_metric_at['global_p95'] = pd.to_numeric(df_metric_at['global_p95'], errors='coerce').fillna(0)
df_metric_at['global_max'] = pd.to_numeric(df_metric_at['global_max'], errors='coerce').fillna(0)
df_metric_at['ratio'] = np.where(
    df_metric_at['global_p95'] == 0,
    np.where(df_metric_at['value_at_16_24'] > 0, np.inf, 1.0),
    df_metric_at['value_at_16_24'] / df_metric_at['global_p95']
)
df_metric_at['breached'] = df_metric_at['value_at_16_24'] > df_metric_at['global_p95']

# Build metric_breaches: per cmdb_id => breached_kpi_count, top_breached_kpi (highest ratio), top_ratio
breaches = df_metric_at[df_metric_at['breached']].copy()
if not breaches.empty:
    top_per_cmdb = breaches.sort_values(['cmdb_id','ratio'], ascending=[True,False]) \
                           .groupby('cmdb_id', as_index=False).first()[['cmdb_id','kpi_name','ratio']]
    metric_breaches = (
        breaches.groupby('cmdb_id', as_index=False)
        .agg(breached_kpi_count=('kpi_name','size'))
        .merge(top_per_cmdb, on='cmdb_id', how='left')
        .rename(columns={'kpi_name':'top_breached_kpi','ratio':'top_ratio'})
    )
else:
    metric_breaches = pd.DataFrame(columns=['cmdb_id','breached_kpi_count','top_breached_kpi','top_ratio'])

# Ensure all allowed cmdbs present
mb_base = pd.DataFrame({'cmdb_id': allowed_cmdbs})
metric_breaches = mb_base.merge(metric_breaches, on='cmdb_id', how='left')
metric_breaches['breached_kpi_count'] = metric_breaches['breached_kpi_count'].fillna(0).astype(int)
metric_breaches['top_breached_kpi'] = metric_breaches['top_breached_kpi'].fillna('NA')
metric_breaches['top_ratio'] = metric_breaches['top_ratio'].where(~metric_breaches['top_ratio'].isna(), None)

# Store metric_breaches in kernel variable for follow-ups (it is assigned)

# --- Step 3: extract directional traces at target timestamp ---
df_trace_slice = df_trace[
    (df_trace['timestamp'] == target_ts) &
    (df_trace['cmdb_id'].isin(allowed_cmdbs)) &
    (df_trace['trace_name'].str.contains(r'\bto_|from_', na=False))
].copy()

# Extract caller_name from trace_name
def extract_caller(trace_name):
    m = re.search(r'(?:to_|from_)([^._]+)', str(trace_name))
    return m.group(1) if m else 'NA'

df_trace_slice['caller_name'] = df_trace_slice['trace_name'].apply(extract_caller)
df_trace_slice['callee_name'] = df_trace_slice['cmdb_id']
df_trace_slice['trace_value'] = df_trace_slice['value']

# Store df_trace_slice in kernel variable for follow-ups (it is assigned)

# --- Step 4: left-join directional trace rows with metric_breaches on callee_name == cmdb_id ---
df_joined = df_trace_slice.merge(metric_breaches, left_on='callee_name', right_on='cmdb_id', how='left', suffixes=('','_m'))

# Prepare output columns per instruction
df_joined['callee_breached_any'] = df_joined['breached_kpi_count'].fillna(0).astype(int) > 0
df_joined['callee_breached_kpi_count'] = df_joined['breached_kpi_count'].fillna(0).astype(int)
df_joined['callee_top_breached_kpi'] = df_joined['top_breached_kpi'].fillna('NA')
df_joined['callee_top_ratio'] = df_joined['top_ratio'].where(~df_joined['top_ratio'].isna(), None)

result = df_joined[['callee_name','caller_name','trace_name','trace_value',
                    'callee_breached_any','callee_breached_kpi_count','callee_top_breached_kpi','callee_top_ratio']].copy()

# --- Step 5: sort by trace_value desc and return top 20 ---
result_sorted = result.sort_values('trace_value', ascending=False).head(20).reset_index(drop=True)

# Summary: which callee components appear most frequently among the top rows and which had breaches
freq = result_sorted.groupby('callee_name').size().reset_index(name='count').sort_values('count', ascending=False)
freq = freq.merge(metric_breaches[['cmdb_id','breached_kpi_count']], left_on='callee_name', right_on='cmdb_id', how='left')
freq['had_breaches'] = freq['breached_kpi_count'].fillna(0).astype(int) > 0
freq_summary_items = [f"{row['callee_name']}({row['count']} rows, breached={row['had_breaches']})" for _, row in freq.iterrows()]
summary_str = "Top callees: " + ", ".join(freq_summary_items) if len(freq_summary_items) > 0 else "No directional traces at this timestamp."

# Final compact outputs: result_sorted (max 20 rows) and summary_str
result_sorted, summary_str

```
Out[9]:
```
Summary of directional traces at 2021-03-09 16:24:00 (UTC+8) joined with metric breaches at 16:24:

Top directional trace rows (highest trace_value first) — shown: callee <- caller : trace_name = trace_value ; callee_breached_any (breached_kpi_count, top_breached_kpi @ top_ratio)
1. Tomcat02 <- IG02 : trace.from_IG02.row_count = 199.0 ; breached=True (7 KPIs, top=OSLinux-...-sdb_DSKBps @ 1.818)
2. IG02 <- Tomcat02 : trace.to_Tomcat02.row_count = 199.0 ; breached=True (2 KPIs, top=OSLinux-CPU_CPU_CPUWio @ 1.470)
3. IG02 <- Tomcat04 : trace.to_Tomcat04.row_count = 191.0 ; breached=True (2 KPIs, top=OSLinux-CPU_CPU_CPUWio @ 1.470)
4. Tomcat04 <- IG02 : trace.from_IG02.row_count = 191.0 ; breached=True (1 KPI, top=Tomcat-Threads..._CurrentThreads @ 1.381)
5. Tomcat02 <- MG02 : trace.to_MG02.row_count = 190.0 ; breached=True (7 KPIs, top=OSLinux-...-sdb_DSKBps @ 1.818)
6. MG02 <- Tomcat02 : trace.from_Tomcat02.row_count = 190.0 ; breached=True (2 KPIs, top=OSLinux-...-TCP-FIN-WAIT @ 2.308)
7. MG02 <- dockerB2 : trace.to_dockerB2.row_count = 182.0 ; breached=True (2 KPIs, top=OSLinux-...-TCP-FIN-WAIT @ 2.308)
8. MG01 <- Tomcat02 : trace.from_Tomcat02.row_count = 181.0 ; breached=False (0 KPIs)
9. Tomcat02 <- MG01 : trace.to_MG01.row_count = 181.0 ; breached=True (7 KPIs, top=OSLinux-...-sdb_DSKBps @ 1.818)
10. Tomcat04 <- MG01 : trace.to_MG01.row_count = 181.0 ; breached=True (1 KPI, top=Tomcat-Threads..._CurrentThreads @ 1.381)
(remaining top-20 rows include MG01/MG02/dockerB1/dockerB2 interactions and IG/ Tomcat1 entries)

Frequency & breach summary (top callees among top-20 rows):
- MG01 appears most often (5 rows) — but had no metric breaches at 16:24 (callee_breached_any=False).
- MG02 appears 4 times — had metric breaches (True).
- Tomcat02 appears 3 times — had metric breaches (True, 7 KPIs).
- Tomcat04 appears 3 times — had metric breaches (True).
- IG02, Tomcat01, IG01 also appear and most had metric breaches.

Interpretation / next steps (concise):
- High directional load/row_count flows concentrate on MG <-> Tomcat and IG -> Tomcat paths at 16:24.
- Many callees with the largest trace values also had metric breaches (notably Tomcat02, Tomcat04, MG02) — priority targets for investigation.
- MG01 shows many heavy incoming/outgoing trace counts but did NOT show metric KPI breaches at 16:24 — investigate whether MG01 is a throughput hub (metrics not captured) or if issues are upstream/downstream.
- Actionable checks: inspect Tomcat02/Tomcat04 and MG02 metrics (disk I/O, TCP FIN/CLOSE_WAIT, thread counts) and correlate with the high trace callers (IG02, Tomcat02, dockerB*) to find root cause of latency/queueing.

The original code execution output of IPython Kernel is also provided below for reference:

(   callee_name caller_name                     trace_name  trace_value  callee_breached_any  callee_breached_kpi_count                            callee_top_breached_kpi  callee_top_ratio
0     Tomcat02        IG02      trace.from_IG02.row_count        199.0                 True                          7     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps          1.818182
1         IG02    Tomcat02    trace.to_Tomcat02.row_count        199.0                 True                          2                             OSLinux-CPU_CPU_CPUWio          1.469765
2         IG02    Tomcat04    trace.to_Tomcat04.row_count        191.0                 True                          2                             OSLinux-CPU_CPU_CPUWio          1.469765
3     Tomcat04        IG02      trace.from_IG02.row_count        191.0                 True                          1  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...          1.380952
4     Tomcat02        MG02        trace.to_MG02.row_count        190.0                 True                          7     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps          1.818182
5         MG02    Tomcat02  trace.from_Tomcat02.row_count        190.0                 True                          2       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          2.307692
6         MG02    dockerB2    trace.to_dockerB2.row_count        182.0                 True                          2       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          2.307692
7         MG01    Tomcat02  trace.from_Tomcat02.row_count        181.0                False                          0                                                 NA               NaN
8     Tomcat02        MG01        trace.to_MG01.row_count        181.0                 True                          7     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps          1.818182
9     Tomcat04        MG01        trace.to_MG01.row_count        181.0                 True                          1  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...          1.380952
10        MG01    Tomcat04  trace.from_Tomcat04.row_count        181.0                False                          0                                                 NA               NaN
11    Tomcat04        MG02        trace.to_MG02.row_count        180.0                 True                          1  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...          1.380952
12        MG02    Tomcat04  trace.from_Tomcat04.row_count        180.0                 True                          2       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          2.307692
13    Tomcat01        MG01        trace.to_MG01.row_count        179.0                 True                          1  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...          1.627907
14        MG01    Tomcat01  trace.from_Tomcat01.row_count        179.0                False                          0                                                 NA               NaN
15        IG01    Tomcat01    trace.to_Tomcat01.row_count        176.0                 True                          1                            OSLinux-CPU_CPU_CPULoad          1.041199
16    Tomcat01        IG01      trace.from_IG01.row_count        176.0                 True                          1  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...          1.627907
17        MG01    dockerB1    trace.to_dockerB1.row_count        175.0                False                          0                                                 NA               NaN
18        MG02    dockerB1    trace.to_dockerB1.row_count        174.0                 True                          2       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT          2.307692
19        MG01    dockerB2    trace.to_dockerB2.row_count        173.0                False                          0                                                 NA               NaN, 'Top callees: MG01(5 rows, breached=False), MG02(4 rows, breached=True), Tomcat02(3 rows, breached=True), Tomcat04(3 rows, breached=True), IG02(2 rows, breached=True), Tomcat01(2 rows, breached=True), IG01(1 rows, breached=True)')```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Candidate components
candidates = ['MG02','MG01','Tomcat02','Tomcat04','apache02','Mysql02','Mysql01','IG01']

# Reuse intermediate kernel tables if available: anomaly_summary (metric), anomaly_summary_trace (trace), agg_table (logs)
# If any are missing, create empty placeholders to avoid errors
if 'anomaly_summary' not in globals():
    anomaly_summary = pd.DataFrame(columns=['cmdb_id','earliest_anomaly_timestamp','earliest_anomaly_ts_local','ratio'])
if 'anomaly_summary_trace' not in globals():
    anomaly_summary_trace = pd.DataFrame(columns=['cmdb_id','earliest_anomaly_timestamp','earliest_anomaly_ts_local','ratio'])
if 'agg_table' not in globals():
    agg_table = pd.DataFrame(columns=['cmdb_id','earliest_error_timestamp_local'])

# Helper: extract metric aggregates per candidate
if not anomaly_summary.empty:
    m = anomaly_summary.copy()
    # prefer existing tz-aware column names; handle both possible column names
    if 'earliest_anomaly_ts_local' in m.columns:
        m['metric_ts'] = pd.to_datetime(m['earliest_anomaly_ts_local'], utc=True, errors='coerce').dt.tz_convert(tz)
    elif 'earliest_anomaly_timestamp' in m.columns:
        m['metric_ts'] = pd.to_datetime(m['earliest_anomaly_timestamp'], unit='s', utc=True, errors='coerce').dt.tz_convert(tz)
    else:
        m['metric_ts'] = pd.NaT
    m['metric_ratio'] = pd.to_numeric(m['ratio'], errors='coerce').fillna(0)
    metric_agg = m.groupby('cmdb_id').agg(
        earliest_metric_anomaly_ts_local=('metric_ts','min'),
        max_metric_ratio=('metric_ratio','max')
    ).reset_index()
else:
    metric_agg = pd.DataFrame(columns=['cmdb_id','earliest_metric_anomaly_ts_local','max_metric_ratio'])

# Helper: extract trace aggregates per candidate
if not anomaly_summary_trace.empty:
    t = anomaly_summary_trace.copy()
    if 'earliest_anomaly_ts_local' in t.columns:
        t['trace_ts'] = pd.to_datetime(t['earliest_anomaly_ts_local'], utc=True, errors='coerce').dt.tz_convert(tz)
    elif 'earliest_anomaly_timestamp' in t.columns:
        t['trace_ts'] = pd.to_datetime(t['earliest_anomaly_timestamp'], unit='s', utc=True, errors='coerce').dt.tz_convert(tz)
    else:
        t['trace_ts'] = pd.NaT
    t['trace_ratio'] = pd.to_numeric(t['ratio'], errors='coerce').fillna(0)
    trace_agg = t.groupby('cmdb_id').agg(
        earliest_trace_anomaly_ts_local=('trace_ts','min'),
        max_trace_ratio=('trace_ratio','max')
    ).reset_index()
else:
    trace_agg = pd.DataFrame(columns=['cmdb_id','earliest_trace_anomaly_ts_local','max_trace_ratio'])

# Helper: extract earliest log error timestamp per candidate from agg_table
# agg_table may contain 'earliest_error_timestamp_local' as string or timestamp; coerce
l = agg_table.copy() if not agg_table.empty else pd.DataFrame(columns=['cmdb_id','earliest_error_timestamp_local'])
if 'earliest_error_timestamp_local' in l.columns:
    # Convert 'NA' and strings to datetime where possible
    def parse_log_ts(x):
        if pd.isna(x) or x == 'NA':
            return pd.NaT
        try:
            return pd.to_datetime(x, utc=True, errors='coerce').tz_convert(tz)
        except Exception:
            # try parsing without timezone, assume already local string like '2021-03-09 16:24:00+08:00'
            try:
                return pd.to_datetime(x, utc=True, errors='coerce').dt.tz_convert(tz)
            except Exception:
                return pd.NaT
    l['log_ts'] = l['earliest_error_timestamp_local'].apply(parse_log_ts)
else:
    l['log_ts'] = pd.NaT
log_agg = l.groupby('cmdb_id').agg(earliest_log_error_ts_local=('log_ts','min')).reset_index()
# has_error_log flag
log_agg['has_error_log'] = ~log_agg['earliest_log_error_ts_local'].isna()

# Build base dataframe for candidates
base = pd.DataFrame({'cmdb_id': candidates})

# Merge aggregates
df = base.merge(metric_agg, on='cmdb_id', how='left') \
         .merge(trace_agg, on='cmdb_id', how='left') \
         .merge(log_agg, on='cmdb_id', how='left')

# Normalize ratio NaNs to None for clarity; timestamps may be NaT
df['max_metric_ratio'] = df['max_metric_ratio'].where(~df['max_metric_ratio'].isna(), None)
df['max_trace_ratio'] = df['max_trace_ratio'].where(~df['max_trace_ratio'].isna(), None)

# Determine earliest_overall_ts and primary_evidence_source with tie-break rules
def choose_primary(row):
    # collect (source, timestamp) pairs
    cand = []
    if not pd.isna(row['earliest_metric_anomaly_ts_local']):
        cand.append(('metric', row['earliest_metric_anomaly_ts_local'], row['max_metric_ratio'] or 0))
    if not pd.isna(row['earliest_trace_anomaly_ts_local']):
        cand.append(('trace', row['earliest_trace_anomaly_ts_local'], row['max_trace_ratio'] or 0))
    if not pd.isna(row['earliest_log_error_ts_local']):
        cand.append(('log', row['earliest_log_error_ts_local'], 0))  # log has no ratio
    if not cand:
        return pd.Series([ 'NA', pd.NaT ])
    # find earliest timestamp
    cand_sorted = sorted(cand, key=lambda x: x[1])
    earliest_time = cand_sorted[0][1]
    # find all sources that have this earliest_time
    tied = [c for c in cand if c[1] == earliest_time]
    if len(tied) == 1:
        return pd.Series([ tied[0][0], earliest_time ])
    # tie-break by larger ratio among tied (log ratio=0)
    max_ratio = max([t[2] for t in tied])
    top_sources = [t[0] for t in tied if t[2] == max_ratio]
    # if still tie, prefer log > trace > metric
    for pref in ['log','trace','metric']:
        if pref in top_sources:
            return pd.Series([ pref, earliest_time ])
    return pd.Series([ top_sources[0], earliest_time ])

df[['primary_evidence_source','earliest_overall_ts_local_raw']] = df.apply(choose_primary, axis=1)

# Format timestamp fields to localized strings or 'NA'
def fmt_ts(x):
    if pd.isna(x):
        return 'NA'
    return x.strftime('%Y-%m-%d %H:%M:%S%z')

df['earliest_metric_anomaly_ts_local'] = df['earliest_metric_anomaly_ts_local'].apply(fmt_ts)
df['earliest_trace_anomaly_ts_local'] = df['earliest_trace_anomaly_ts_local'].apply(fmt_ts)
df['earliest_log_error_ts_local'] = df['earliest_log_error_ts_local'].apply(fmt_ts)
df['earliest_overall_ts_local'] = df['earliest_overall_ts_local_raw'].apply(fmt_ts)

# Ensure numeric ratios display NA if None
df['max_metric_ratio'] = df['max_metric_ratio'].apply(lambda x: round(x,6) if pd.notna(x) and x is not None else 'NA')
df['max_trace_ratio'] = df['max_trace_ratio'].apply(lambda x: round(x,6) if pd.notna(x) and x is not None else 'NA')

# Select final columns in requested order
out_cols = [
    'cmdb_id',
    'earliest_metric_anomaly_ts_local',
    'max_metric_ratio',
    'earliest_trace_anomaly_ts_local',
    'max_trace_ratio',
    'earliest_log_error_ts_local',
    'primary_evidence_source',
    'earliest_overall_ts_local'
]
out_df = df[out_cols].copy()

# Sort by earliest_overall_ts_local ascending (NA last)
# convert earliest_overall_ts_local_raw to datetime for sorting
out_df['_sort_ts'] = pd.to_datetime(df['earliest_overall_ts_local_raw'], utc=True, errors='coerce')
out_df = out_df.sort_values(by=['_sort_ts','cmdb_id'], ascending=[True, True], na_position='last').drop(columns=['_sort_ts'])

# Limit to 8 rows and display compact textual table
out_df = out_df.head(8).reset_index(drop=True)
out_df

```
Out[10]:
```
Compact confirmation for requested components (window 2021-03-09 16:00–16:30 UTC+8). One line per component: cmdb_id — earliest_metric_anomaly (local) — max_metric_ratio — earliest_trace_anomaly (local) — max_trace_ratio — earliest_log_error (local) — primary_evidence — earliest_overall.

- Mysql01 — 2021-03-09 16:00:00+08:00 — ratio=inf — NA — NA — NA — primary: metric — earliest overall: 2021-03-09 16:00:00+08:00  
- Mysql02 — 2021-03-09 16:00:00+08:00 — ratio=inf — NA — NA — NA — primary: metric — earliest overall: 2021-03-09 16:00:00+08:00  
- Tomcat02 — 2021-03-09 16:00:00+08:00 — ratio=1.818182 — 2021-03-09 16:02:00+08:00 — ratio=1.848441 — NA — primary: metric — earliest overall: 2021-03-09 16:00:00+08:00  
- Tomcat04 — 2021-03-09 16:00:00+08:00 — ratio=20.000000 — 2021-03-09 16:00:00+08:00 — ratio=2.582278 — NA — primary: metric — earliest overall: 2021-03-09 16:00:00+08:00  
- MG02 — 2021-03-09 16:01:00+08:00 — ratio=10.000000 — 2021-03-09 16:02:00+08:00 — ratio=3.565292 — NA — primary: metric — earliest overall: 2021-03-09 16:01:00+08:00  
- apache02 — 2021-03-09 16:01:00+08:00 — ratio=inf — NA — NA — 2021-03-09 16:24:00+08:00 — primary: metric — earliest overall: 2021-03-09 16:01:00+08:00  
- IG01 — 2021-03-09 16:02:00+08:00 — ratio=3.333333 — 2021-03-09 16:02:00+08:00 — ratio=2.837052 — NA — primary: metric — earliest overall: 2021-03-09 16:02:00+08:00  
- MG01 — 2021-03-09 16:02:00+08:00 — ratio=10.000000 — 2021-03-09 16:02:00+08:00 — ratio=2.843698 — NA — primary: metric — earliest overall: 2021-03-09 16:02:00+08:00

Concise interpretation and recommended single root-cause pick:
- Earliest anomalies begin at ~16:00 (Mysql01, Mysql02, Tomcat02, Tomcat04). Tomcat04 shows the largest metric spike by ratio (20x historical p95) at 16:00 and concurrent trace anomaly at 16:00 — strong candidate for primary root cause.  
- Mysql01 and Mysql02 have infinite ratios (global_p95==0 with nonzero values) at 16:00 — these are critical outliers and deserve immediate attention as well.  
- MG01 and MG02 show very large metric ratios (~10x) slightly later (~16:01–16:02) and strong trace ratios ~2.8–3.6 around ~16:02, indicating propagation.  
- apache02 has an infinite metric ratio at 16:01 and an error log entry at 16:24 (correlation point), but its earliest signal is metric at 16:01.

Recommendation to select a single component to start with:
- Start with Tomcat04 (first 16:00 anomaly + highest metric ratio), while immediately triaging Mysql01/Mysql02 anomalies in parallel (inf ratios indicate unusual or newly observed metrics). Then investigate MG01/MG02 and apache02 for propagation and logs around 16:01–16:24.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id earliest_metric_anomaly_ts_local  max_metric_ratio earliest_trace_anomaly_ts_local max_trace_ratio earliest_log_error_ts_local primary_evidence_source earliest_overall_ts_local
0   Mysql01         2021-03-09 16:00:00+0800               inf                              NA              NA                          NA                  metric  2021-03-09 16:00:00+0800
1   Mysql02         2021-03-09 16:00:00+0800               inf                              NA              NA                          NA                  metric  2021-03-09 16:00:00+0800
2  Tomcat02         2021-03-09 16:00:00+0800          1.818182        2021-03-09 16:02:00+0800        1.848441                          NA                  metric  2021-03-09 16:00:00+0800
3  Tomcat04         2021-03-09 16:00:00+0800         20.000000        2021-03-09 16:00:00+0800        2.582278                          NA                  metric  2021-03-09 16:00:00+0800
4      MG02         2021-03-09 16:01:00+0800         10.000000        2021-03-09 16:02:00+0800        3.565292                          NA                  metric  2021-03-09 16:01:00+0800
5  apache02         2021-03-09 16:01:00+0800               inf                              NA              NA    2021-03-09 16:24:00+0800                  metric  2021-03-09 16:01:00+0800
6      IG01         2021-03-09 16:02:00+0800          3.333333        2021-03-09 16:02:00+0800        2.837052                          NA                  metric  2021-03-09 16:02:00+0800
7      MG01         2021-03-09 16:02:00+0800         10.000000        2021-03-09 16:02:00+0800        2.843698                          NA                  metric  2021-03-09 16:02:00+0800```
```